In [ ]:
import numpy as np
import json
import pandas as pd
# from tensorflow.keras.metrics import MeanSquaredError
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import ConvLSTM2D, BatchNormalization, Conv3D
from sklearn.model_selection import KFold 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from scipy.stats import kendalltau
import seaborn as sns
from tqdm import tqdm
import os

In [ ]:
lookback = 1
forecast_horizon = 1

In [ ]:

#Where the entire dataset split into its timestamps is stored
timestamps_directory = 'split_files_cleaned/'

#Where the list of all timestamps are stored
timestamps_file_path = os.path.join(timestamps_directory, 'alltimestamps_cleaned.json')

#Where the individual samples are stored
saved_files = 'lookback_and_lookahead_files_cleaned/'

#Where the list of timestamps in their splits are stored
split_file = 'timestamps_splits_cleaned2.npz'

In [ ]:
# Utility to load all timestamps
def load_all_timestamps():
    with open(timestamps_file_path, 'r') as file:
        timestamps = json.load(file)
        sorted_timestamps = sorted(timestamps)
        return sorted_timestamps

# Utilities to find lengths
def load_split_lengths():
    loaded_data = np.load(split_file)
    train_len = len(loaded_data['train'])
    val_len = len(loaded_data['val'])
    return train_len, train_len + val_len  # Train length and cumulative Train+Val length

def get_all_lengths():
    loaded_data = np.load(split_file)
    train_len = len(loaded_data['train'])
    val_len = len(loaded_data['val'])
    test_len = len(loaded_data['test'])
    return train_len, val_len, test_len

# Function for loading a single training sample
def load_singular_train_data(index, lookback):
    all_timestamps = load_all_timestamps()
    timestamp_name = all_timestamps[lookback + index]
    file_name = f'{index + lookback}_{timestamp_name}.npz'
    file_path = os.path.join(saved_files, file_name)
    
    data = np.load(file_path, allow_pickle=True)
    return data['X_batches'], data['y_batches']

# Function for loading a single validation sample
def load_singular_val_data(index, lookback):
    all_timestamps = load_all_timestamps()
    train_len, _ = load_split_lengths()
    timestamp_name = all_timestamps[lookback + train_len + index]
    file_name = f'{index + train_len + lookback}_{timestamp_name}.npz'
    file_path = os.path.join(saved_files, file_name)
    
    data = np.load(file_path, allow_pickle=True)
    return data['X_batches'], data['y_batches']

# Function for loading a single test sample
def load_singular_test_data(index, lookback):
    all_timestamps = load_all_timestamps()
    train_len, train_val_len = load_split_lengths()
    timestamp_name = all_timestamps[lookback + train_val_len + index]
    file_name = f'{index + train_val_len + lookback}_{timestamp_name}.npz'
    file_path = os.path.join(saved_files, file_name)
    
    try:
        data = np.load(file_path, allow_pickle=True)
    except:
        print("The last erroneous files don't exist")

    return data['X_batches'], data['y_batches']

In [ ]:
def fill_none_with_mean(values):
    array = np.array([np.nan if v is None else v for v in values])
    nan_indices = np.isnan(array)
    non_nan_indices = np.where(~nan_indices)[0]
    non_nan_values = array[non_nan_indices]
    if len(non_nan_values) == 0: 
        return np.zeros_like(array).tolist()
    array[nan_indices] = np.interp(np.where(nan_indices)[0], non_nan_indices, non_nan_values)
    return array.tolist()

In [ ]:
#Implementation of the TimeSeriesDataset

from torch.utils.data import Dataset, DataLoader

class TimeSeriesDataset(Dataset):
    def __init__(self, indices, lookback, mode="train"):
        """
        The parameters are: 
        indices is a list of indices, 
        lookback is set manually, 
        mode to indicate how we are appropriately adding the indices.
        
        """
        self.indices = indices
        self.lookback = lookback
        self.mode = mode
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        index = self.indices[idx]

        # Load data based on mode
        if self.mode == "train":
            X, y = load_singular_train_data(index, self.lookback)
        elif self.mode == "val":
            X, y = load_singular_val_data(index, self.lookback)
        elif self.mode == "test":
            X, y = load_singular_test_data(index, self.lookback)

        for i, array in enumerate(X):
            for j, sub_array in enumerate(array):
                X[i][j] = fill_none_with_mean(sub_array)
        
        X = np.array(X)  

        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.float32)
        
        return X_tensor, y_tensor


In [ ]:
#Getting lengths of the training, validation and testing
train_length, val_length, test_length = get_all_lengths()

In [ ]:
# Split data indices for train, validation, and test
train_length, val_length, test_length = get_all_lengths()

#From 0 up until the length of the training, validation and test sets 

train_indices = list(range(0, train_length))
val_indices = list(range(0, val_length))
test_indices = list(range(0, test_length))

# Initialize Datasets
train_dataset = TimeSeriesDataset(train_indices, lookback, mode="train")
val_dataset = TimeSeriesDataset(val_indices, lookback, mode="val")
test_dataset = TimeSeriesDataset(test_indices, lookback, mode="test")

#Initialize DataLoaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
#Normalization functions
import json

with open('globaldatastatistics.json', 'r') as f:
    global_stats = json.load(f)

# Function to normalize data using global statistics
def normalize_global(batch, variable_name):
    mean = global_stats[variable_name]['mean']
    std = global_stats[variable_name]['std']
    return (batch - mean) / (std)

# Function to normalize data using local statistics
def normalize_local(batch):
    mean = batch.mean()  # Compute mean across the time dimension (axis=1)
    std = batch.std()# Compute std deviation across time dimension
    return (batch - mean) / (std)    #Normalize batch and leave a small epsilon to avoid division by 0

def normalize_precipitation(batch):
    log_normalized = torch.log(batch + 1)
    zero_indicator = (batch == 0).float()
    return log_normalized, zero_indicator


In [ ]:
#Variable names
variable_names = ['10 metre U wind component', '10 metre V wind component', '2 metre dewpoint temperature', '2 metre temperature', 'Total column rain water', 'Total precipitation', 'Surface latent heat flux']

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from DConvLSTM_SAC import DConvLSTM_SAC

device_ = 'cuda' if torch.cuda.is_availabile() else 'cpu'

# Configuration class
class Configs:
    def __init__(self):
        self.patch_size = 1  # No patching for raw data
        self.img_channel = 1  # Number of input channels (atmospheric variables)
        self.img_width = 81  # Grid width (latitudes)
        self.img_height = 97  # Grid height (longitudes)
        self.filter_size = 3  # Convolution kernel size
        self.stride = 1
        self.layer_norm = True  # Whether to use layer normalization
        self.num_layers = 3 
        self.num_hidden = [64, 64, 64]  
        self.batch_size = 16 
        self.input_length = 1  # Number of input time steps
        self.total_length = 2  # Input + output time steps (predict 1 step ahead)
        self.device = device_

configs = Configs()

# Initialize the model
model = DConvLSTM_SAC(
    num_layers=configs.num_layers,
    num_hidden=configs.num_hidden,
    configs=configs
).to(configs.device)

# Optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.MSELoss()

# Print model summary
print(model)

optimizer = optim.AdamW(model.parameters(), lr = 0.005) 
loss_fn = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)

train_losses = []
val_losses = []

y_true = []
y_pred = []

train_losses_per_fold = []
val_losses_per_fold = []

height = 81
width = 97

num_epochs = 400
scaling_factor = 1



In [ ]:
#Setting device
if torch.cuda.is_available():
    print("running on cuda")
    device = torch.device('cuda')
else:
    print("running on the cpu")
    device = torch.device('cpu')

In [ ]:
def generate_mask_true(batch_size, total_length, input_length, img_width, img_height, prob=1.0, decay_rate=0.95, epoch=0):
    """
    Generate a decaying probabilistic mask for scheduled sampling.

    Args:
        batch_size (int): Batch size.
        total_length (int): Total number of time steps (input + prediction).
        input_length (int): Number of input time steps.
        img_width (int): Spatial grid width.
        img_height (int): Spatial grid height.
        prob (float): Initial probability of using ground truth.
        decay_rate (float): Decay rate per epoch.
        epoch (int): Current training epoch.

    Returns:
        mask_true (torch.Tensor): Probabilistic mask for scheduled sampling.
    """
    current_prob = prob * (decay_rate ** epoch)
    mask = torch.rand(batch_size, total_length - input_length, img_width, img_height)
    mask_true = (mask < current_prob).float()  # 1 if below probability, else 0
    return mask_true


In [ ]:

def spatial_correlation(y_true, y_pred):
    # Flatten the tensors to work with them
    y_true_flat = y_true.view(-1).cpu()
    y_pred_flat = y_pred.view(-1).cpu()

    # Compute the numerator: sum(P * T)
    numerator = torch.sum(y_pred_flat * y_true_flat)

    #Compute the denominator: sqrt(sum(P^2) * sum(T^2))
    denominator = torch.sqrt(torch.sum(y_pred_flat ** 2) * torch.sum(y_true_flat ** 2))

    #compute the correlation (add epsilon to avoid division by zero)
    correlation = numerator / (denominator)

    return correlation.item()

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from libpysal.weights import lat2W
from esda.moran import Moran_Local

def compute_local_morans_i(data, height, width):
    """
    Compute the Local Moran's I index for each spatial grid.
    Args:
        data: A 2D NumPy array of shape (height, width) representing the spatial data.
        height: Height of the spatial grid.
        width: Width of the spatial grid.
    Returns:
        morans_i: A 2D NumPy array of shape (height, width) containing Local Moran's I values.
    """
    w = lat2W(height, width)
    flattened_data = data.flatten()
    local_morans = Moran_Local(flattened_data, w, permutations=0)
    return local_morans.Is.reshape(height, width)

def preprocess_data(data_loader, variable_names, global_stats, name, configs):
    """
    Preprocess the dataset to:
    1. Apply log normalization for precipitation.
    2. Compute the Local Moran's I index for precipitation (inputs and targets).
    3. Format the data for the DConvLSTM_SAC model.
    4. Crop width to ensure data is 81x81 (square grid).
    """
    height = configs.img_width  # Assume height remains 81
    width = configs.img_height  #Original width is 97
    cropped_width = 81          #Final cropped width
    processed_batches = []

    count = 0

    for X_batch, y_batch in tqdm(data_loader, desc=f"Preprocessing {name.capitalize()} Data"):
        X_batch = X_batch.clone()
        y_batch = y_batch.clone()

        if count == len(data_loader) - 2:
            break 

        count +=1

        #Index of precipitation in the input variables
        precip_idx = variable_names.index('Total precipitation')

        #Scale precipitation (X_batch and y_batch) and apply log normalization
        X_batch[:, :, precip_idx] *= 1000  #Convert to mm
        y_batch *= 1000
        X_batch[:, :, precip_idx] = torch.log(X_batch[:, :, precip_idx] + 1e-3)
        y_batch = torch.log(y_batch + 1e-3)

        #Reshape for spatial grid processing
        X_batch = X_batch.reshape(configs.batch_size, configs.input_length, len(variable_names), height, width)
        y_batch = y_batch.reshape(configs.batch_size, 1, 1, height, width)

        #Crop width to 81x81
        X_batch = X_batch[:, :, :, :, 8:-8]  #remove first 8 and last 8 columns
        y_batch = y_batch[:, :, :, :, 8:-8]

        #Reshape target precipitation to match time dimension of input
        y_batch_expanded = y_batch.expand(-1, configs.input_length, -1, -1, -1)  #Match input_length for time dimension

        # Combine precipitation (X_batch and y_batch) for Moran's computation
        combined_precip = torch.cat((X_batch[:, :, precip_idx:precip_idx+1], y_batch_expanded), dim=2)
        # combined_precip shape: [batch_size, input_length, 2, height, cropped_width]

        # Compute Moran's for combined precipitation
        morans_i_batch = []
        for t in range(combined_precip.size(1)):
            morans_i_timestep = []
            for b in range(combined_precip.size(0)): 
                grid_data = combined_precip[b, t, 0].view(height, cropped_width).cpu().numpy()  # Input precipitation
                morans_input = compute_local_morans_i(grid_data, height, cropped_width)

                grid_data_target = combined_precip[b, t, 1].view(height, cropped_width).cpu().numpy()  # Target precipitation
                morans_target = compute_local_morans_i(grid_data_target, height, cropped_width)

                # Stack Moran's indices for input and target precipitation
                morans_i_timestep.append(np.stack([morans_input, morans_target]))

            # Stack all batches for this timestep
            morans_i_batch.append(np.stack(morans_i_timestep))

        morans_i_batch = torch.tensor(np.stack(morans_i_batch), dtype=torch.float32).to(X_batch.device)
        morans_i_batch = morans_i_batch.view(configs.batch_size, combined_precip.size(1), 2, height, cropped_width)

        print(morans_i_batch.shape)

        combined_precip = combined_precip.view(combined_precip.size(0), combined_precip.size(1), 2, height, cropped_width)

        # Concatenate Moran's I with precipitation
        X_precip_with_moran = torch.cat((combined_precip, morans_i_batch), dim=2)  # Shape: [batch, time, 3, height, width]

        # Add remaining atmospheric variables to the frames tensor
        X_other = X_batch[:, :, [i for i in range(X_batch.size(2)) if i != precip_idx]].view(
            X_batch.size(0), X_batch.size(1), -1, height, cropped_width
        )
        frames_tensor = torch.cat((X_precip_with_moran, X_other), dim=2)  # Combine all features

        # Ensure frames_tensor matches expected shape: [batch_size, total_length, height, width, frame_channel]
        frames_tensor = frames_tensor.permute(0, 1, 3, 4, 2).contiguous()  # Reorder dimensions

        # Generate mask_true (default all ones for now, can be modified later for scheduled sampling)
        mask_true = torch.ones(
            (X_batch.size(0), configs.total_length - configs.input_length, height, cropped_width, frames_tensor.size(-1))
        ).float().to(X_batch.device)

        # Append to processed batches
        processed_batches.append((frames_tensor, mask_true))

    return processed_batches


# # Precompute normalized data for train, validation, and test
normalized_train_data = preprocess_data(train_loader, variable_names, global_stats, "train", configs)
normalized_val_data = preprocess_data(val_loader, variable_names, global_stats, "val", configs)
normalized_test_data = preprocess_data(test_loader, variable_names, global_stats, "test", configs)






In [ ]:
# import torch

# Save the preprocessed data
torch.save(normalized_train_data, "normalized_train_data_DconvSAC.pt")
torch.save(normalized_val_data, "normalized_val_data_DconvSAC.pt")
torch.save(normalized_test_data, "normalized_test_data_DconvSAC.pt")


In [ ]:
# Moran's index anomalies calculating

import matplotlib.pyplot as plt
import torch
import numpy as np
from tqdm import tqdm
from libpysal.weights import lat2W
from esda.moran import Moran_Local

def compute_temporal_morans(data_loader, variable_names, height=81, width=97):
    """
    Compute the mean Moran's I for each timestep across all batches in the data loader.
    Args:
        data_loader: DataLoader containing normalized input data.
        variable_names: List of variable names in the dataset.
        height, width: Dimensions of the spatial grid.
    Returns:
        temporal_morans: List of mean Moran's I values for each timestep.
    """
    temporal_morans = []
    count = 0

    for X_batch, _ in tqdm(data_loader, desc="Computing Moran's Index"):

        if count == len(data_loader)-2:
            break

        precip_idx = variable_names.index("Total precipitation")
        X_precip = X_batch[:, :, precip_idx]  # Extract precipitation variable

        count += 1

        batch_morans = []  # Moran's I for each batch
        for t in range(X_precip.size(1)): 
            timestep_morans = []
            for b in range(X_precip.size(0)):  # Loop over batch
                grid_data = X_precip[b, t].cpu().numpy().reshape(height, width)
                morans_i = compute_local_morans_i(grid_data, height, width)
                timestep_morans.append(np.mean(morans_i))  
            batch_morans.append(np.mean(timestep_morans))  # Average over batch
        temporal_morans.extend(batch_morans)

    return temporal_morans


variable_names = ['10 metre U wind component', '10 metre V wind component', '2 metre dewpoint temperature', '2 metre temperature', 'Total column rain water', 'Total precipitation', 'Surface latent heat flux']
temporal_morans = compute_temporal_morans(test_loader, variable_names)

# Plot Moran's Index over time
plt.figure(figsize=(10, 6))
plt.plot(temporal_morans, label="Moran's I Over Time")
plt.axhline(np.mean(temporal_morans), color='red', linestyle='--', label="Mean Moran's I")
plt.xlabel("Timestep")
plt.ylabel("Moran's I")
plt.title("Temporal Variation of Moran's I for Precipitation")
plt.legend()
plt.show()

#Detect anomalies based on Moran's indes
mean_morans = np.mean(temporal_morans)
std_morans = np.std(temporal_morans)

anomalies = [i for i, mi in enumerate(temporal_morans) if abs(mi - mean_morans) > 2 * std_morans]

print(f"Anomalies detected at timesteps: {anomalies}")


In [ ]:
#Definition of all the evaluation functions

from scipy.stats import pearsonr, spearmanr
import torch.nn.functional as F
from scipy.spatial.distance import pdist, squareform
import numpy as np
import torch

# Nash-Sutcliffe Efficiency
def nash_sutcliffe_efficiency(observed, predicted):
    observed = observed.cpu()
    predicted = predicted.cpu()
    numerator = torch.sum((observed - predicted) ** 2)
    denominator = torch.sum((observed - torch.mean(observed)) ** 2)
    return 1 - (numerator / denominator).item()

# Pearson Correlation
def pearson_correlation(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()
    y_pred = y_pred.view(-1).cpu().numpy()
    return pearsonr(y_true, y_pred)[0]

# Spearman Correlation
def spearman_correlation(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()
    y_pred = y_pred.view(-1).cpu().numpy()
    return spearmanr(y_true, y_pred).correlation

# Mean Squared Error
def mse(y_true, y_pred):
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    return torch.mean((y_true - y_pred) ** 2).item()

# Mean Absolute Error
def mae(y_true, y_pred):
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    return torch.mean(torch.abs(y_true - y_pred)).item()

# Percentage Error
def percentage_error(y_true, y_pred):
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    return 100 * torch.mean((y_pred - y_true) / (y_true + 1e-6)).item()

# Percentage Bias
def percentage_bias(y_true, y_pred):
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    return 100 * torch.sum(y_pred - y_true) / (torch.sum(y_true) + 1e-6)

# Earth Mover's Distance
def earth_movers_distance(y_true, y_pred):
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    emd = torch.mean(torch.abs(torch.sort(y_pred)[0] - torch.sort(y_true)[0])).item()
    return emd

# Kendall Tau
def kendall_tau(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()
    y_pred = y_pred.view(-1).cpu().numpy()
    from scipy.stats import kendalltau
    return kendalltau(y_true, y_pred).correlation

# R2 Score
def r2_score(y_true, y_pred):
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    ss_total = torch.sum((y_true - torch.mean(y_true)) ** 2)
    ss_residual = torch.sum((y_true - y_pred) ** 2)
    return 1 - (ss_residual / (ss_total + 1e-6)).item()

# Spatial Correlation
def spatial_correlation(y_true, y_pred):
    y_true_flat = y_true.view(-1).cpu()
    y_pred_flat = y_pred.view(-1).cpu()
    numerator = torch.sum(y_pred_flat * y_true_flat)
    denominator = torch.sqrt(torch.sum(y_pred_flat ** 2) * torch.sum(y_true_flat ** 2))
    return (numerator / denominator).item()

# Root Mean Squared Error (RMSE)
def rmse(y_true, y_pred):
    return torch.sqrt(mse(y_true, y_pred)).item()

def variogram_function(grid_values, grid_shape, lag_distance):
    """
    Compute the variogram directly for grid-based spatial data.

    Parameters:
        grid_values (np.ndarray): Flattened values (1D array).
        grid_shape (tuple): Shape of the grid (e.g., (81, 97)).
        lag_distance (int): Distance in grid cells for the lag.

    Returns:
        float: Variogram value.
    """
    grid_values = grid_values.reshape(grid_shape)
    height, width = grid_shape
    variogram_sum = 0
    count = 0

    for row in range(height):
        for col in range(width):
            # Neighbor in the horizontal direction
            if col + lag_distance < width:
                diff = grid_values[row, col] - grid_values[row, col + lag_distance]
                variogram_sum += diff ** 2
                count += 1

            # Neighbor in the vertical direction
            if row + lag_distance < height:
                diff = grid_values[row, col] - grid_values[row + lag_distance, col]
                variogram_sum += diff ** 2
                count += 1

    return variogram_sum / (2 * count) if count > 0 else 0



In [ ]:
import numpy as np

def compute_morans_index_with_significance(values, spatial_weights, num_permutations=1000, grid_shape=(81, 97)):
    """
    compute moran's i and test statistical significance using permutation testing

    parameters:
        values (np.ndarray): flattened 1D array of grid values, or stacked batch of grids
        spatial_weights (np.ndarray): spatial adjacency/weight matrix of shape (n, n), where n = h * w
        num_permutations (int): number of permutations for null distribution (default = 1000)
        grid_shape (tuple): spatial dimensions of the data grid (height, width)

    returns:
        tuple: (moran's i, z-score, p-value)
    """

    n = grid_shape[0] * grid_shape[1]  # total number of spatial units
    batch_size = values.size // n      # how many grid maps are stacked (e.g. in time or batches)

    # reshape input values to shape (batch, H, W)
    values = values.reshape(batch_size, grid_shape[0], grid_shape[1])

    # check if the spatial weights matrix is the correct shape
    if spatial_weights.shape != (n, n):
        raise ValueError(
            f"Spatial weights matrix shape {spatial_weights.shape} does not match expected size ({n}, {n})."
        )

    # average values across batch dimension (i.e. mean grid)
    values_meaned = np.mean(values, axis=0).flatten()  # flatten to (n,)

    x_mean = np.mean(values_meaned)                   # global mean of the grid
    deviations = values_meaned - x_mean               # mean-centered values

    # compute numerator of moran's i: sum(w_ij * (x_i - x̄) * (x_j - x̄))
    numerator = np.sum(spatial_weights * np.outer(deviations, deviations))

    # compute denominator: sum((x_i - x̄)^2)
    denominator = np.sum(deviations ** 2)

    # compute moran's i using the formula: I = (n / sum(w)) * (numerator / denominator)
    morans_i = (n / np.sum(spatial_weights)) * (numerator / denominator)

    # ------------------------
    # permutation testing to create null distribution
    # ------------------------
    null_distribution = []
    for _ in range(num_permutations):
        # randomly permute the values to destroy spatial structure
        permuted_values = np.random.permutation(values_meaned)

        # mean-centered permuted values
        perm_deviations = permuted_values - x_mean

        # recompute numerator with permuted data
        perm_numerator = np.sum(spatial_weights * np.outer(perm_deviations, perm_deviations))

        # compute permuted moran's i
        perm_morans_i = (n / np.sum(spatial_weights)) * (perm_numerator / denominator)

        null_distribution.append(perm_morans_i)

    null_distribution = np.array(null_distribution)

    # compute mean and standard deviation of the null distribution
    null_mean = np.mean(null_distribution)
    null_std = np.std(null_distribution)

    # compute z-score: how many std deviations the observed value is from the null mean
    z_score = (morans_i - null_mean) / null_std

    # compute two-tailed p-value from the null distribution
    p_value = 2 * min(
        np.mean(null_distribution >= morans_i),  # probability of getting something more positive
        np.mean(null_distribution <= morans_i)   # probability of getting something more negative
    )

    return morans_i, z_score, p_value


In [ ]:
# Load the preprocessed data
normalized_train_data = torch.load("normalized_train_data_DconvSAC.pt")
normalized_val_data = torch.load("normalized_val_data_DconvSAC.pt")
normalized_test_data = torch.load("normalized_test_data_DconvSAC.pt")


In [ ]:
# Training and Validation Loop
best_val_loss = float('inf')
best_val_spatial_corr = -float('inf')
best_y_pred = None
best_y_true = None
all_y_true = []
all_y_pred = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    running_spatial_corr = 0.0
    spatial_corr_count = 0

    for frames_tensor, mask_true in tqdm(normalized_train_data, desc=f'Epoch {epoch+1}/{num_epochs} Training'):
        # Move data to the appropriate device
        frames_tensor = frames_tensor.to(configs.device)
        mask_true = mask_true.to(configs.device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(frames_tensor, mask_true)


        # Extract predictions and ground truth
        y_pred = outputs[:, :, :, :, 0]  # Predicted precipitation
        y_true = frames_tensor[:, :, :, :, 0]  # Ground truth precipitation

        # Compute regression loss
        loss = loss_fn(y_pred, y_true)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * frames_tensor.size(0)

        # Compute spatial correlation
        batch_spatial_corr = spatial_correlation(y_true, y_pred)
        running_spatial_corr += batch_spatial_corr
        spatial_corr_count += 1

    train_loss = running_loss / len(normalized_train_data)
    train_spatial_corr = running_spatial_corr / spatial_corr_count

    print(f"Epoch {epoch + 1}/{num_epochs} Training Loss: {train_loss:.6f}")
    print(f"Epoch {epoch + 1}/{num_epochs} Training Spatial Correlation: {train_spatial_corr:.6f}")

    model.eval()
    val_loss = 0.0
    running_val_spatial_corr = 0.0
    val_corr_count = 0

    with torch.no_grad():
        for frames_tensor, mask_true in tqdm(normalized_val_data, desc=f'Epoch {epoch+1}/{num_epochs} Validation'):
            # Move data to the appropriate device
            frames_tensor = frames_tensor.to(configs.device)
            mask_true = mask_true.to(configs.device)

            # Forward pass
            outputs = model(frames_tensor, mask_true)

            # Extract predictions and ground truth
            y_pred = outputs[:, configs.input_length:, :, :, 0]
            y_true = frames_tensor[:, :, :, :, 0]

            all_y_pred.append(y_pred.cpu())
            all_y_true.append(y_true.cpu())

            # Compute validation loss
            loss = loss_fn(y_pred, y_true)
            val_loss += loss.item() * frames_tensor.size(0)

            # Compute spatial correlation
            spatial_corr = spatial_correlation(y_true, y_pred)
            running_val_spatial_corr += spatial_corr
            val_corr_count += 1

        val_loss /= len(normalized_val_data)
        avg_val_spatial_corr = running_val_spatial_corr / val_corr_count

        print(f"Epoch {epoch + 1}/{num_epochs} Validation Loss: {val_loss:.6f}")
        print(f"Epoch {epoch + 1}/{num_epochs} Validation Spatial Correlation: {avg_val_spatial_corr:.6f}")

    # Save best validation results
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_y_pred = y_pred
        best_y_true = y_true

    if avg_val_spatial_corr > best_val_spatial_corr:
        best_val_spatial_corr = avg_val_spatial_corr

    scheduler.step(val_loss)

all_y_pred = torch.cat(all_y_pred, dim=0)
all_y_true = torch.cat(all_y_true, dim=0)

torch.save({
    'y_true': best_y_true,
    'y_pred': best_y_pred,
    'all_y_true': all_y_true,
    'all_y_pred': all_y_pred,
}, 'best_validation_results_SAC_DConvLSTM.pth')

print("Best validation predictions and ground truth saved.")

model_path = "SAC_DConvLSTM_model.pth"
torch.save(model.state_dict(), model_path)



In [ ]:
# Save the best validation predictions and true values
torch.save({
    'y_true_loss': best_y_true,
    'y_pred_loss': best_y_pred,
}, 'best_validation_results_SAC_ConvLSTM.pth')

print("Best validation predictions and ground truth saved.")

In [ ]:
# Save the model state dictionary
model_path = "SAC_DConvLSTM_model.pth"
torch.save({
    'model_state_dict': model.state_dict(),
}, model_path)


In [ ]:
# Compute Moran's on Validation Data

spatial_weights = np.identity(height*width)
model.eval()
all_morans_indices = []

with torch.no_grad():
    for X_val, y_val in tqdm(normalized_val_data, desc='Computing Moran\'s I on Validation Data'):
        X_val = X_val.to(device)
        y_val = y_val.to(device)

        batch_size, time_steps_in, channels_in, grid_points = X_val.shape
        X_val = X_val.view(batch_size, time_steps_in, channels_in, height, width)

        sac_features = X_val[:, :, -1, :, :]  #extract Moran's I
        sac_features = sac_features.unsqueeze(2)
        X_val = X_val[:, :, :-1, :, :]  # Remove Moran's index from input features

        regression_output = model(X_val, sac_features=sac_features)

        regression_output_flat = regression_output.view(-1).detach().cpu().numpy()
        morans_i, z_score, p_value = compute_morans_index_with_significance(regression_output_flat, spatial_weights)
        all_morans_indices.append(morans_i)

print(f"Mean Moran's I on Validation Data: {np.mean(all_morans_indices):.6f}")


In [ ]:
#Visualise the training and validation losses

import matplotlib.pyplot as plt

# Plot the training and validation loss for all of the folds combined
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label="Train Loss")
plt.xlabel("Epochs")
plt.ylabel("Training Dice Loss")
plt.title("Training Loss")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epochs")
plt.ylabel("Validation Loss")
plt.title("Validation Loss")
plt.legend()
plt.show()

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from DConvLSTM_SAC import DConvLSTM_SAC

device_ = 'cuda' if torch.cuda.is_availabile() else 'cpu'

# Configuration class
class Configs:
    def __init__(self):
        self.patch_size = 1  # No patching for raw data
        self.img_channel = 1  # Number of input channels (atmospheric variables)
        self.img_width = 81  # Grid width (latitudes)
        self.img_height = 97  # Grid height (longitudes)
        self.filter_size = 3  # Convolution kernel size
        self.stride = 1
        self.layer_norm = True  # Whether to use layer normalization
        self.num_layers = 3 
        self.num_hidden = [64, 64, 64]  
        self.batch_size = 16 
        self.input_length = 1  # Number of input time steps
        self.total_length = 2  # Input + output time steps (predict 1 step ahead)
        self.device = device_

configs = Configs()

# Initialize the model
model = DConvLSTM_SAC(
    num_layers=configs.num_layers,
    num_hidden=configs.num_hidden,
    configs=configs
).to(configs.device)

# Optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.MSELoss()

# Print model summary
print(model)

optimizer = optim.AdamW(model.parameters(), lr = 0.005) 
loss_fn = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)

height = 81
width = 97


In [ ]:
# Load saved predictions and ground truth
import json

with open('globaldatastatistics.json', 'r') as f:
    global_stats = json.load(f)
    
best_validation_predictions_and_ground_truth = torch.load('best_validation_results_SAC_DConvLSTM.pth')

# Regression outputs
y_true = best_validation_predictions_and_ground_truth['y_true_corr']
y_pred = best_validation_predictions_and_ground_truth['y_pred_corr']

# Ensure data is on the CPU and in numpy format
y_pred_original = y_pred
y_true_original = y_true

# Evaluate metrics in the original scale
metrics = {
    "R2": r2_score(y_true_original, y_pred_original),
    "NSE": nash_sutcliffe_efficiency(y_true_original, y_pred_original),
    "MSE": mse(y_true_original, y_pred_original),
    "MAE": mae(y_true_original, y_pred_original),
    "Pearson": pearson_correlation(y_true_original, y_pred_original),
    "Spearman": spearman_correlation(y_true_original, y_pred_original),
    "Kendall Tau": kendall_tau(y_true_original, y_pred_original),
    "Percentage Error": percentage_error(y_true_original, y_pred_original),
    "Percentage Bias": percentage_bias(y_true_original, y_pred_original),
    "EMD": earth_movers_distance(y_true_original, y_pred_original),
    "Spatial Correlation": spatial_correlation(y_true_original, y_pred_original),
}

# Print regression metrics
print("\nRegression Metrics (Original Scale):")
for metric, value in metrics.items():
    print(f"{metric}: {value:.6f}")

# Print mean and standard deviation
print("\nMean and SD of Ground Truth Precipitation (Original Scale):")
print(f"Mean: {y_true_mean:.6f}, SD: {y_true_std:.6f}")

print("\nMean and SD of Predicted Precipitation (Original Scale):")
print(f"Mean: {y_pred_mean:.6f}, SD: {y_pred_std:.6f}")



In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch

best_validation_predictions_and_ground_truth = torch.load('best_validation_results_SAC_DConvLSTM.pth')

y_true = best_validation_predictions_and_ground_truth['y_true_loss']
y_pred = best_validation_predictions_and_ground_truth['y_pred_loss']

y_true = y_true.cpu().numpy()
y_pred = y_pred.cpu().numpy()

y_pred_original = y_pred
y_true_original = y_true


def plot_precipitation_distribution(precipitation_values, title):
    """
    Plot the histogram of precipitation values.
    """
    plt.figure(figsize=(10, 6))
    plt.hist(precipitation_values, bins=50, edgecolor='black', log=True)
    plt.title(f'Precipitation Distribution for {title}')
    plt.xlabel('Precipitation (mm)')
    plt.ylabel('Frequency (log scale)')
    plt.show()

def plot_scatter(y_true, y_pred, title):
    """
    Scatter plot of predicted vs ground truth precipitation.
    """
    plt.figure(figsize=(8, 6))
    plt.scatter(y_true, y_pred, alpha=0.5, s=10, c='blue')
    plt.xlabel("Ground Truth")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.show()

def plot_spatial_heatmap(data, title):
    """
    Heatmap for spatial precipitation data.
    """
    mean_precip = np.mean(data, axis=0) 
    plt.figure(figsize=(10, 8))
    sns.heatmap(mean_precip, cmap="coolwarm", cbar=True)
    plt.title(title)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.show()

def plot_morans_index(morans_indices, title="Moran's I over time"):
    """
    Plot Moran's I values over time.
    """
    plt.figure(figsize=(10, 6))
    plt.plot(morans_indices, marker='o', linestyle='-', label="Moran's I")
    plt.title(title)
    plt.xlabel('Time Steps')
    plt.ylabel("Moran's I")
    plt.grid()
    plt.legend()
    plt.show()

def plot_variogram(variogram_values, title="Variogram Function over Time"):
    """
    Plot the variogram function values over time.
    """
    plt.figure(figsize=(10, 6))
    plt.plot(variogram_values, marker='x', linestyle='-', color='red', label='Variogram (r)')
    plt.title(title)
    plt.xlabel('Time Steps')
    plt.ylabel('Variogram (r)')
    plt.grid()
    plt.legend()
    plt.show()

y_true_flat = y_true_original.flatten()
y_pred_flat = y_pred_original.flatten()

plt.figure(figsize=(8, 6))
hb = plt.hexbin(y_true_flat, y_pred_flat, gridsize=50, bins='log', cmap='plasma', mincnt=1)
plt.colorbar(hb, label='Count')
plt.xlabel("Ground Truth")
plt.ylabel("Predicted")
plt.title("Hexbin plot of ground truth vs predicted precipitation")
plt.show()

plot_precipitation_distribution(y_true_flat, "Ground Truth Precipitation")
plot_precipitation_distribution(y_pred_flat, "Predicted Precipitation")

plot_scatter(y_true_flat, y_pred_flat, "Scatter Plot of Ground Truth vs Predicted Precipitation")

plot_spatial_heatmap(y_true_original[0, 0], "Ground Truth Spatial Precipitation (First Timestamp)")
plot_spatial_heatmap(y_pred_original[0, 0], "Predicted Spatial Precipitation (First Timestamp)")


morans_indices = best_validation_predictions_and_ground_truth.get('morans_indices', [])
if morans_indices:
    plot_morans_index(morans_indices)

variogram_values = best_validation_predictions_and_ground_truth.get('variogram_values', [])
if variogram_values:
    plot_variogram(variogram_values)





In [ ]:
#morans index and variogram take and extremely long time to run so are commented out in the following

def evaluate(model, test_loader, reg_loss_fn, device, variable_names, height, width):
    """
    Evaluate the model on the test set for regression tasks with additional metrics
    like Moran's Index and Variogram function (r).
    """
    model.eval() 
    test_reg_loss = 0.0

    y_true_reg = [] 
    y_pred_reg = []  
    morans_indices = []  
    variogram_values = []  
    # Disable gradient computation
    with torch.no_grad():
        for X_test, y_test in tqdm(test_loader, desc="Evaluating on Test Set"):
            # Move the batch to the device
            X_test, y_test = X_test.to(device), y_test.to(device)

            # Reshape inputs and targets
            batch_size, time_steps_in, channels_in, grid_points = X_test.shape
            batch_size, time_steps_out, channels_out, grid_points = y_test.shape
            X_test = X_test.view(batch_size, time_steps_in, channels_in, height, width)
            y_test = y_test.view(batch_size, time_steps_out, channels_out, height, width)

            sac_features = X_test[:, :, -1, :, :]
            sac_features = sac_features.unsqueeze(2)
            X_test = X_test[:, :, :-1, :, :]  # Remove Moran's I from the main input

            regression_output = model(X_val, sac_features=sac_features)

            # Forward pass
            regression_output = model(X_test, sac_features=sac_features)

            # Compute regression loss
            reg_loss = reg_loss_fn(regression_output, y_test)

            # Accumulate losses
            test_reg_loss += reg_loss.item() * X_test.size(0)

            # Collect true and predicted values for regression
            y_true_reg.append(y_test.cpu())
            y_pred_reg.append(regression_output.cpu())

            # # Compute Moran's 
            # morans_index = compute_morans_index_with_significance(regression_output)
            # morans_indices.append(morans_index)

            # # Compute Variogram
            # variogram_r = variogram_function(y_test, regression_output)
            # variogram_values.append(variogram_r)

    # Normalize regression loss by the total dataset size
    test_reg_loss /= len(test_loader)

    print(f"Test Regression Loss: {test_reg_loss:.16f}")

    y_true_reg_flat = torch.cat(y_true_reg, dim=0).flatten()  
    y_pred_reg_flat = torch.cat(y_pred_reg, dim=0).flatten() 

    # Compute regression metrics
    regression_metrics = {
        "MSE": mse(y_true_reg_flat, y_pred_reg_flat),
        "MAE": mae(y_true_reg_flat, y_pred_reg_flat),
        "NSE": nash_sutcliffe_efficiency(y_true_reg_flat, y_pred_reg_flat),
        "R2": r2_score(y_true_reg_flat, y_pred_reg_flat),
        "Pearson": pearson_correlation(y_true_reg_flat, y_pred_reg_flat),
        "Spearman": spearman_correlation(y_true_reg_flat, y_pred_reg_flat),
        "Percentage Error": percentage_error(y_true_reg_flat, y_pred_reg_flat),
        "Percentage Bias": percentage_bias(y_true_reg_flat, y_pred_reg_flat),
        "EMD": earth_movers_distance(y_true_reg_flat, y_pred_reg_flat),
        "Kendall Tau": kendall_tau(y_true_reg_flat, y_pred_reg_flat),
        "Spatial Correlation": spatial_correlation(y_true_reg_flat, y_pred_reg_flat),
        # "Moran's I (Mean)": np.mean(morans_indices),
        # "Variogram (r)": np.mean(variogram_values),
    }

    print("\nRegression Metrics:")
    for metric, value in regression_metrics.items():
        print(f"{metric}: {value:.16f}")

    torch.save({
        'y_true_reg': y_true_reg_flat,
        'y_pred_reg': y_pred_reg_flat,
        'morans_indices': morans_indices,
        'variogram_values': variogram_values,
    }, 'SAC_DConvLSTM_testing_results.pth')

    return test_reg_loss, regression_metrics

# Run the evaluation
test_total_loss, regression_metrics = evaluate(
    model=model,
    test_loader=normalized_test_data,
    reg_loss_fn=loss_fn,
    device=device,
    variable_names=variable_names,
    height=height,
    width=width,
)


In [ ]:

results = torch.load('SAC_DConvLSTM_testing_results.pth')

y_true_reg = results['y_true_reg']
y_pred_reg = results['y_pred_reg']
morans_indices = results['morans_indices']
variogram_values = results['variogram_values']

regression_metrics = {
    "MSE": mse(y_true_reg, y_pred_reg),
    "MAE": mae(y_true_reg, y_pred_reg),
    "NSE": nash_sutcliffe_efficiency(y_true_reg, y_pred_reg),
    "R2": r2_score(y_true_reg, y_pred_reg),
    "Pearson": pearson_correlation(y_true_reg, y_pred_reg),
    "Spearman": spearman_correlation(y_true_reg, y_pred_reg),
    "Percentage Error": percentage_error(y_true_reg, y_pred_reg),
    "Percentage Bias": percentage_bias(y_true_reg, y_pred_reg),
    "EMD": earth_movers_distance(y_true_reg, y_pred_reg),
    "Kendall Tau": kendall_tau(y_true_reg, y_pred_reg),
    "Spatial Correlation": spatial_correlation(y_true_reg, y_pred_reg),
    "Moran's I (Mean)": torch.tensor(morans_indices).mean().item(),
    "Variogram (r)": torch.tensor(variogram_values).mean().item(),
}



print("\nRegression Metrics:")
for metric, value in regression_metrics.items():
    print(f"{metric}: {value:.16f}")

In [ ]:
# Visualize Moran's 
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(morans_indices, bins=20, kde=True, color='blue')
plt.title("Distribution of Moran's Index")
plt.xlabel("Moran's Index")
plt.ylabel("Frequency")
plt.show()

plt.figure(figsize=(10, 6))
sns.histplot(variogram_values, bins=20, kde=True, color='green')
plt.title("Distribution of Variogram (r)")
plt.xlabel("Variogram (r)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import torch

best_validation_predictions_and_ground_truth = torch.load('best_validation_results_DSAC_ConvLSTM.pth')

y_true = best_validation_predictions_and_ground_truth['y_true_loss']
y_pred = best_validation_predictions_and_ground_truth['y_pred_loss']

y_true = y_true.cpu().numpy()
y_pred = y_pred.cpu().numpy()

def plot_precipitation_distribution(precipitation_values, title):
    """
    Plot the histogram of precipitation values.
    """
    plt.figure(figsize=(10, 6))
    plt.hist(precipitation_values, bins=50, edgecolor='black', log=True)
    plt.title(f'Precipitation Distribution for {title}')
    plt.xlabel('Precipitation (mm)')
    plt.ylabel('Frequency (log scale)')
    plt.show()

def plot_scatter(y_true, y_pred, title):
    """
    Scatter plot of predicted vs ground truth precipitation.
    """
    plt.figure(figsize=(8, 6))
    plt.scatter(y_true, y_pred, alpha=0.5, s=10, c='blue')
    plt.xlabel("Ground Truth")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.show()

def plot_spatial_heatmap(data, title):
    """
    Heatmap for spatial precipitation data.
    """
    mean_precip = np.mean(data, axis=0) 
    plt.figure(figsize=(10, 8))
    sns.heatmap(mean_precip, cmap="coolwarm", cbar=True)
    plt.title(title)
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.show()

def plot_morans_index(morans_indices, title="Moran's I over time"):
    """
    Plot Moran's I values over time.
    """
    plt.figure(figsize=(10, 6))
    plt.plot(morans_indices, marker='o', linestyle='-', label="Moran's I")
    plt.title(title)
    plt.xlabel('Time Steps')
    plt.ylabel("Moran's I")
    plt.grid()
    plt.legend()
    plt.show()

def plot_variogram(variogram_values, title="Variogram Function over Time"):
    """
    Plot the variogram function values over time.
    """
    plt.figure(figsize=(10, 6))
    plt.plot(variogram_values, marker='x', linestyle='-', color='red', label='Variogram (r)')
    plt.title(title)
    plt.xlabel('Time Steps')
    plt.ylabel('Variogram (r)')
    plt.grid()
    plt.legend()
    plt.show()

y_true_flat = y_true.flatten()
y_pred_flat = y_pred.flatten()

# Visualizations
plt.figure(figsize=(8, 6))
hb = plt.hexbin(y_true_flat, y_pred_flat, gridsize=50, bins='log', cmap='plasma', mincnt=1)
plt.colorbar(hb, label='Count')
plt.xlabel("Ground Truth")
plt.ylabel("Predicted")
plt.title("Hexbin plot of ground truth vs predicted precipitation")
plt.show()


plot_precipitation_distribution(y_true_flat, "Ground Truth Precipitation")
plot_precipitation_distribution(y_pred_flat, "Predicted Precipitation")

plot_scatter(y_true_flat, y_pred_flat, "Scatter Plot of Ground Truth vs Predicted Precipitation")

plot_spatial_heatmap(y_true[0, 0], "Ground Truth Spatial Precipitation (First Timestamp)")
plot_spatial_heatmap(y_pred[0, 0], "Predicted Spatial Precipitation (First Timestamp)")


morans_indices = best_validation_predictions_and_ground_truth.get('morans_indices', [])
if morans_indices:
    plot_morans_index(morans_indices)


variogram_values = best_validation_predictions_and_ground_truth.get('variogram_values', [])
if variogram_values:
    plot_variogram(variogram_values)




In [ ]:

results = torch.load('SAC_DConvLSTM_testing_results.pth')

y_true_reg = results['y_true_reg']
y_pred_reg = results['y_pred_reg']
morans_indices = results['morans_indices']
variogram_values = results['variogram_values']

In [ ]:
def plot_predictions(y_true_reg, y_pred_reg, morans_indices, batch_index, num_samples, height=81, width=97, batch_size=16):
    """
    Visualize the predictions, residuals, and Moran's I for a given batch and selected samples.

    - y_true_reg: Flattened ground truth tensor for regression.
    - y_pred_reg: Flattened predicted tensor for regression.
    - morans_indices: List of Moran's I values for each batch.
    - batch_index: Index of the batch to visualize.
    - num_samples: Number of samples from the batch to visualize.
    - height: Height of the grid.
    - width: Width of the grid.
    - batch_size: Number of samples per batch.
    """
    # Compute total samples per batch (height * width * batch_size)
    samples_per_batch = height * width * batch_size

    # Extract the current batch from flattened arrays
    start_idx = batch_index * samples_per_batch
    end_idx = start_idx + samples_per_batch

    true_batch_reg = y_true_reg[start_idx:end_idx].reshape(batch_size, height, width)
    pred_batch_reg = y_pred_reg[start_idx:end_idx].reshape(batch_size, height, width)

    # Compute residuals (
    residuals = true_batch_reg - pred_batch_reg

    #Plot specified number of samples from the batch
    for i in range(min(num_samples, batch_size)):
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        # Plot true regression values
        im1 = axes[0].imshow(true_batch_reg[i], cmap='viridis')
        axes[0].set_title(f"Batch {batch_index}, Sample {i} - Ground Truth (Regression)")
        plt.colorbar(im1, ax=axes[0])

        # Plot predicted regression values
        im2 = axes[1].imshow(pred_batch_reg[i], cmap='viridis')
        axes[1].set_title(f"Batch {batch_index}, Sample {i} - Predicted (Regression)")
        plt.colorbar(im2, ax=axes[1])

        # Plot regression residuals
        residuals_np = residuals[i] 
        im3 = axes[2].imshow(
            residuals_np,
            cmap='RdBu',
            vmin=-np.abs(residuals_np).max(),  
            vmax=np.abs(residuals_np).max()
        )
        axes[2].set_title(f"Batch {batch_index}, Sample {i} - Residuals (True - Pred)")
        plt.colorbar(im3, ax=axes[2])

        plt.tight_layout()
        plt.show()

    # Plot Moran's values for the batch
    plt.figure(figsize=(10, 6))
    plt.plot(range(len(morans_indices)), morans_indices, marker='o', linestyle='-', color='blue')
    plt.title("Moran's I for Predicted Data")
    plt.xlabel("Batch Index")
    plt.ylabel("Moran's I")
    plt.axhline(y=0, color='red', linestyle='--', label="No Spatial Autocorrelation")
    plt.legend()
    plt.grid()
    plt.show()


plot_predictions(y_true_reg, y_pred_reg, morans_indices, batch_index=0, num_samples=5)

